In [2]:
print(agents["registered_on"].dtype)
print(agents["registered_on"].head())

object
0    2025-04-09
1    2025-11-14
2    2024-10-13
3    2024-10-09
4    2025-01-18
Name: registered_on, dtype: object


In [5]:
# connect and pull the tables
import os
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine

load_dotenv()
engine = create_engine(os.getenv("DATABASE_URL"))

agents = pd.read_sql("SELECT * FROM agents", engine)
transactions = pd.read_sql("SELECT * FROM transactions", engine)

# make sure these come through as real dates, not text
agents["registered_on"] = pd.to_datetime(agents["registered_on"])
transactions["txn_ts"] = pd.to_datetime(transactions["txn_ts"])


# cohort month per agent, same as DATE_TRUNC('month', registered_on) in sql
agents["cohort_month"] = agents["registered_on"].dt.to_period("M").dt.to_timestamp()

# how many agents per cohort
cohort_sizes = agents.groupby("cohort_month").size().reset_index(name="cohort_size")


# attach each agent's cohort month onto their transactions
# validate=many_to_one checks each txn matches exactly one agent, same
# assumption the sql join was relying on
txn_with_cohort = transactions.merge(
    agents[["agent_id", "cohort_month"]],
    on="agent_id",
    how="inner",
    validate="many_to_one",
)

txn_with_cohort["activity_month"] = txn_with_cohort["txn_ts"].dt.to_period("M").dt.to_timestamp()


# one row per agent per month they were active, duplicates dropped
# same as the DISTINCT in the sql version
activity = txn_with_cohort[["agent_id", "cohort_month", "activity_month"]].drop_duplicates()


# months since registration, same math as the sql EXTRACT/month_number calc
activity["month_number"] = (
    (activity["activity_month"].dt.year - activity["cohort_month"].dt.year) * 12
    + (activity["activity_month"].dt.month - activity["cohort_month"].dt.month)
)

activity_filtered = activity[activity["month_number"].between(1, 6)]


# count distinct active agents per cohort per month offset
retained = (
    activity_filtered.groupby(["cohort_month", "month_number"])["agent_id"]
    .nunique()
    .reset_index(name="active_agents")
)

pandas_retention = retained.merge(cohort_sizes, on="cohort_month")
pandas_retention["retention_rate"] = (
    pandas_retention["active_agents"] / pandas_retention["cohort_size"]
).round(4)
pandas_retention = pandas_retention.sort_values(
    ["cohort_month", "month_number"]
).reset_index(drop=True)


# same query as sql/analytics.sql Q1, run here so we can diff against it
sql_retention = pd.read_sql("""
    WITH cohorts AS (
        SELECT
            agent_id,
            DATE_TRUNC('month', registered_on) AS cohort_month
        FROM agents
    ),
    cohort_sizes AS (
        SELECT
            cohort_month,
            COUNT(*) AS cohort_size
        FROM cohorts
        GROUP BY cohort_month
    ),
    agent_activity_months AS (
        SELECT DISTINCT
            c.agent_id,
            c.cohort_month,
            DATE_TRUNC('month', t.txn_ts) AS activity_month
        FROM cohorts c
        JOIN transactions t ON t.agent_id = c.agent_id
    ),
    month_offsets AS (
        SELECT
            agent_id,
            cohort_month,
            (EXTRACT(YEAR FROM activity_month) - EXTRACT(YEAR FROM cohort_month)) * 12
              + (EXTRACT(MONTH FROM activity_month) - EXTRACT(MONTH FROM cohort_month)) AS month_number
        FROM agent_activity_months
    ),
    retained AS (
        SELECT
            cohort_month,
            month_number,
            COUNT(DISTINCT agent_id) AS active_agents
        FROM month_offsets
        WHERE month_number BETWEEN 1 AND 6
        GROUP BY cohort_month, month_number
    )
    SELECT
        r.cohort_month,
        r.month_number,
        r.active_agents,
        s.cohort_size,
        ROUND(r.active_agents::numeric / s.cohort_size, 4) AS retention_rate
    FROM retained r
    JOIN cohort_sizes s ON s.cohort_month = r.cohort_month
    ORDER BY r.cohort_month, r.month_number
""", engine)

# drop timezone so this matches the pandas-built cohort_month column
sql_retention["cohort_month"] = sql_retention["cohort_month"].dt.tz_localize(None)


# line up both results and check they agree
compare = pandas_retention.merge(
    sql_retention,
    on=["cohort_month", "month_number"],
    suffixes=("_py", "_sql"),
)
compare["retention_diff"] = (
    compare["retention_rate_py"].astype(float) - compare["retention_rate_sql"].astype(float)
)

mismatches = compare[compare["retention_diff"].abs() > 0.0001]

print("Rows compared:", len(compare))
print("Mismatched rows:", len(mismatches))
print(mismatches)

Rows compared: 0
Mismatched rows: 0
Empty DataFrame
Columns: [cohort_month, month_number, active_agents_py, cohort_size_py, retention_rate_py, active_agents_sql, cohort_size_sql, retention_rate_sql, retention_diff]
Index: []
